# Stage 01 — Paper Intelligence

Read the paper PDF and raw data, then write `data/paper_spec.json`.
Follow `skills/stage_01.md` for detailed instructions.

In [1]:
from paths import *
import json, yaml
import pandas as pd
import pyreadstat

# Load config
config = yaml.safe_load((PROJECT_ROOT / 'config.yaml').read_text())
print('Project root:', PROJECT_ROOT)
print('Config loaded:', list(config.keys()))

Project root: C:\Users\qgallea\Dropbox\work\claude_code\causal_ml_extension\papers\01_out_of_africa
Config loaded: ['paper', 'identification', 'dml', 'review']


In [2]:
# --- AGENT: inspect raw data variable names ---
# Load the primary dataset and print all columns with their Stata labels
primary_file = RAW_DATA_DIR / config['identification']['primary_data_file']
df, meta = pyreadstat.read_dta(str(primary_file))
print(f'Shape: {df.shape}')
print('\nColumn labels:')
for col, label in meta.column_names_to_labels.items():
    print(f'  {col:30s}  {label}')

Shape: (208, 179)

Column labels:
  id                              Numerical country identifier
  code                            World Bank country code
  country                         World Bank country name
  pop1                            Population in 1 CE
  pop1000                         Population in 1000 CE
  pop1500                         Population in 1500 CE
  pd1                             Population density in 1 CE
  pd1000                          Population density in 1000 CE
  pd1500                          Population density in 1500 CE
  ur1500                          Urbanization rate in 1500 CE
  gdppc2000                       Income per capita in 2000 CE
  trust                           Interpersonal trust
  articles                        Scientific articles
  adiv                            Observed genetic diversity
  adiv_sqr                        Observed genetic diversity square
  pdiv                            Predicted genetic diversity
  pdiv_s

In [3]:
# --- AGENT: read PDF and extract paper intelligence ---
# Populated by Stage 01 agent from raw_data/paper.pdf and raw_data/country.dta

paper_spec = {
    "title": "The 'Out of Africa' Hypothesis, Human Genetic Diversity, and Comparative Economic Development",
    "authors": ["Ashraf, Q.", "Galor, O."],
    "year": 2013,
    "journal": "American Economic Review",
    "slug": "ashrafgalor2013",

    "identification": {
        "type": "IV",
        "narrative": (
            "The paper argues that prehistoric migratory distance from East Africa "
            "(Addis Ababa) shaped genetic diversity through the serial founder effect, "
            "whereby sub-groups carrying only a subset of parental genetic diversity "
            "established new settlements farther away. Genetic diversity has a "
            "hump-shaped (quadratic) effect on economic development: too little "
            "diversity (Native American populations) and too much diversity (African "
            "populations) are both detrimental, while intermediate levels (European "
            "and Asian populations) are most conducive to growth. The IV strategy "
            "instruments observed/predicted genetic diversity with migratory distance "
            "from East Africa, which is plausibly exogenous to contemporary development "
            "outcomes. The main contemporary outcome is log income per capita in 2000 CE; "
            "the historical outcome is log population density in 1500 CE."
        ),
        "outcome_variable": "ln_gdppc2000",
        "outcome_label": "Log income per capita in 2000 CE",
        "treatment_variable": "pdiv_aa",
        "treatment_label": "Predicted genetic diversity (ancestry adjusted)",
        "treatment_variable_sq": "pdiv_aa_sqr",
        "treatment_label_sq": "Predicted genetic diversity (ancestry adjusted) square",
        "instrument": "mdist_addis",
        "instrument_label": "Migratory distance from East Africa (Addis Ababa)",
        "functional_form": "quadratic",
        "controls": [
            "ln_yst",       # Log Neolithic transition timing
            "ln_arable",    # Log percentage of arable land
            "ln_abslat",    # Log absolute latitude
            "ln_soilsuit",  # Log soil fertility / land suitability for agriculture
            "abslat",       # Absolute latitude (used in some specs)
            "temp",         # Temperature
            "precip"        # Precipitation
        ],
        "fixed_effects": ["continent"],
        "cluster_se": None,
        "se_notes": "Bootstrap SE (generated regressor); Conley (1999) spatial SE for limited-sample historical regressions; heteroskedasticity-robust SE elsewhere",
        "primary_data_file": "country.dta",
        "secondary_datasets": ["ethnic.dta", "ethnicpair.dta"],
        "historical_outcome_variable": "ln_pd1500",
        "historical_outcome_label": "Log population density in 1500 CE",
        "config_mismatches": {
            "outcome_variable": "config.yaml says 'ln_gdp_pc'; actual column is 'ln_gdppc2000'",
            "treatment_variable": "config.yaml says 'pdiv_pdum'; actual column is 'pdiv_aa' (ancestry-adjusted predicted diversity used in contemporary analysis) or 'pdiv' (unadjusted); 'pdiv_pdum' does not exist",
            "landlocked": "config.yaml lists 'landlocked' as control; column does not exist in country.dta",
            "island": "config.yaml lists 'island' as control; column does not exist in country.dta",
            "ln_land_area": "config.yaml lists 'ln_land_area' as control; column does not exist in country.dta (use 'area' or 'ln_arable' instead)"
        }
    },

    "main_results": [
        {
            "table": "Table 1",
            "spec": "OLS col 4 - observed diversity + all controls (21-country limited sample, historical)",
            "outcome": "ln_pd1500",
            "treatment": "adiv (observed genetic diversity)",
            "coef_linear": 225.440,
            "se_linear": 73.781,
            "coef_sq": -161.158,
            "se_sq": 56.155,
            "n_obs": 21,
            "r_squared": 0.89,
            "optimal_diversity": 0.699,
            "notes": "Controls: log Neolithic transition timing, log arable land, log absolute latitude, log land suitability. Heteroskedasticity-robust SE."
        },
        {
            "table": "Table 3",
            "spec": "OLS col 5 - predicted diversity + all controls (145-country extended sample, historical)",
            "outcome": "ln_pd1500",
            "treatment": "pdiv (predicted genetic diversity)",
            "coef_linear": 195.416,
            "se_linear": 55.916,
            "coef_sq": -137.977,
            "se_sq": 40.773,
            "n_obs": 145,
            "r_squared": 0.67,
            "optimal_diversity": 0.708,
            "notes": "Controls: log Neolithic transition timing, log arable land, log absolute latitude, log land suitability. Bootstrap SE. Continent FE: No."
        },
        {
            "table": "Table 3",
            "spec": "OLS col 6 - predicted diversity + all controls + continent FE (145-country, historical, BASELINE)",
            "outcome": "ln_pd1500",
            "treatment": "pdiv (predicted genetic diversity)",
            "coef_linear": 199.727,
            "se_linear": 80.281,
            "coef_sq": -146.167,
            "se_sq": 56.251,
            "n_obs": 145,
            "r_squared": 0.69,
            "optimal_diversity": 0.683,
            "notes": "Baseline regression equation (8). Controls: log Neolithic transition timing, log arable land, log absolute latitude, log land suitability. Continent FE: Yes. Bootstrap SE."
        },
        {
            "table": "Table 6",
            "spec": "OLS col 1 - ancestry-adjusted diversity + baseline controls + continent FE (143-country, contemporary)",
            "outcome": "ln_gdppc2000",
            "treatment": "pdiv_aa (ancestry-adjusted predicted genetic diversity)",
            "coef_linear": 203.443,
            "se_linear": 83.368,
            "coef_sq": -142.663,
            "se_sq": 59.037,
            "n_obs": 143,
            "r_squared": 0.57,
            "optimal_diversity": 0.713,
            "notes": "Controls: log Neolithic transition timing (unadj.), log arable land, log absolute latitude, log land suitability. Continent FE: Yes. Bootstrap SE."
        },
        {
            "table": "Table 7",
            "spec": "OLS col 5 - ancestry-adjusted diversity + full controls + continent/OPEC/legal/religion FE (109-country, contemporary, BASELINE)",
            "outcome": "ln_gdppc2000",
            "treatment": "pdiv_aa (ancestry-adjusted predicted genetic diversity)",
            "coef_linear": 281.173,
            "se_linear": 70.459,
            "coef_sq": -195.010,
            "se_sq": 49.764,
            "n_obs": 109,
            "r_squared": 0.90,
            "optimal_diversity": 0.721,
            "notes": "Controls: log Neolithic transition timing, log arable land, log absolute latitude, social infrastructure, ethnic fractionalization, malaria risk, tropical zones, waterway distance. Continent + OPEC + legal origin + religion FE. Bootstrap SE. Preferred contemporary specification."
        },
        {
            "table": "Table 2",
            "spec": "2SLS col 5 - observed diversity instrumented by migratory distance (21-country, historical)",
            "outcome": "ln_pd1500",
            "treatment": "adiv instrumented by mdist_addis",
            "coef_linear": 285.190,
            "se_linear": 88.064,
            "coef_sq": -206.576,
            "se_sq": 66.852,
            "n_obs": 21,
            "r_squared": None,
            "optimal_diversity": None,
            "notes": "Controls: log Neolithic transition timing, log arable land, log absolute latitude, log land suitability. Instrument: migratory distance from East Africa (linear + square). Heteroskedasticity-robust SE."
        }
    ],

    "dml": {
        "model_type": "PLIV",
        "rationale": "The paper's main identification is IV (migratory distance instruments for genetic diversity), with a quadratic treatment; PLIV (Partial Linear IV) accommodates the continuous instrument and nonlinear treatment while using ML for the high-dimensional set of geographic, institutional, and cultural controls."
    }
}

print("paper_spec keys:", list(paper_spec.keys()))
print("main_results count:", len(paper_spec["main_results"]))
print("treatment variable (correct):", paper_spec["identification"]["treatment_variable"])
print("outcome variable (correct):", paper_spec["identification"]["outcome_variable"])


paper_spec keys: ['title', 'authors', 'year', 'journal', 'slug', 'identification', 'main_results', 'dml']
main_results count: 6
treatment variable (correct): pdiv_aa
outcome variable (correct): ln_gdppc2000


In [4]:
# --- Write paper_spec.json (atomic write) ---
import tempfile, shutil, os

tmp = PAPER_SPEC.with_suffix('.json.tmp')
tmp.write_text(json.dumps(paper_spec, indent=2))
shutil.move(str(tmp), str(PAPER_SPEC))
print(f'✓ Written: {PAPER_SPEC}')
print(json.dumps(paper_spec, indent=2)[:500])

✓ Written: C:\Users\qgallea\Dropbox\work\claude_code\causal_ml_extension\papers\01_out_of_africa\data\paper_spec.json
{
  "title": "The 'Out of Africa' Hypothesis, Human Genetic Diversity, and Comparative Economic Development",
  "authors": [
    "Ashraf, Q.",
    "Galor, O."
  ],
  "year": 2013,
  "journal": "American Economic Review",
  "slug": "ashrafgalor2013",
  "identification": {
    "type": "IV",
    "narrative": "The paper argues that prehistoric migratory distance from East Africa (Addis Ababa) shaped genetic diversity through the serial founder effect, whereby sub-groups carrying only a subset of par
